In [0]:
from pyspark.sql.functions import current_timestamp

# Caminho da Landing onde estão os CSVs originais
landing_path = "/Volumes/workspace/default/landing"

# Catalog e schema de destino da camada Bronze
catalog = "workspace"
bronze_schema = "bronze"

print(f"Landing: {landing_path}")
print(f"Destino Bronze: {catalog}.{bronze_schema}")


In [0]:
display(dbutils.fs.ls(landing_path))


In [0]:
arquivos_bronze = {
    "movies_info_TMDB_IMDB.csv": "tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "tb_credits_and_tags",
    "movies_reviews.csv": "tb_movies_reviews"
}

arquivos_bronze


In [0]:
# Lê os CSVs e grava as tabelas da camada Bronze

for arquivo, tabela in arquivos_bronze.items():
    caminho_arquivo = f"{landing_path}/{arquivo}"

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(caminho_arquivo)
    )

    df_bronze = df.withColumn(
        "ingestion_datetime",
        current_timestamp()
    )

    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalog}.{bronze_schema}.{tabela}")
    )

    print(f"{arquivo} -> {catalog}.{bronze_schema}.{tabela}")
    

In [0]:
display(spark.sql("SHOW TABLES IN workspace.bronze"))


In [0]:
# Os widgets permitem informar o período da consulta sem precisar alterar diretamente o código do notebook.

from datetime import datetime, timedelta

data_fim_padrao = datetime.now()
data_inicio_padrao = data_fim_padrao - timedelta(days=7)

data_inicio_default = data_inicio_padrao.strftime("%m-%d-%Y")
data_fim_default = data_fim_padrao.strftime("%m-%d-%Y")

dbutils.widgets.text("data_inicio", data_inicio_default, "Data início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", data_fim_default, "Data fim (MM-DD-AAAA)")

data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

print(f"Período consultado: {data_inicio} até {data_fim}")


In [0]:
# Busca a cotação na API do Banco Central ou no JSON da Landing

import requests
import json

url_cotacao = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio}'"
    f"&@dataFinalCotacao='{data_fim}'"
    "&$format=json"
)

try:
    response = requests.get(url_cotacao, timeout=10)
    response.raise_for_status()

    dados_cotacao = response.json()["value"]
    print("Cotação obtida diretamente pela API do Banco Central.")

except requests.RequestException:
    caminho_json = f"{landing_path}/cotacao_dolar.json"

    # Lê o arquivo pelo caminho do Volume no Databricks
    texto_json = (
        spark.read
        .option("wholetext", "true")
        .text(caminho_json)
        .first()["value"]
    )

    dados_cotacao = json.loads(texto_json)["value"]
    print("API indisponível. Cotação carregada do JSON da Landing.")

print(f"Registros encontrados: {len(dados_cotacao)}")


In [0]:
# Converte os dados da cotação para um DataFrame Spark

df_cotacao = spark.createDataFrame(dados_cotacao)

df_cotacao_bronze = df_cotacao.withColumn(
    "ingestion_datetime",
    current_timestamp()
)

display(df_cotacao_bronze)


In [0]:
# Salva os dados da cotação na camada Bronze em formato Delta

(
    df_cotacao_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bronze.tb_cotacao_dolar")
)

print("Tabela tb_cotacao_dolar gravada com sucesso.")


In [0]:
# Confere todas as tabelas criadas na camada Bronze

display(spark.sql("SHOW TABLES IN workspace.bronze"))

